In [ ]:
# Workflow

# 1. Local: run `youtube_audio_downloader` → `./data` (raw audio)
# 2. For large datasets (e.g., 100+ hours), distribute full length audio files across `part1`, `part2`, ... under `./data` to reduce Colab timeout/disconnect risks
# 3. Upload `./data` to Google Drive (`PROJECT_DIR/data`)
# 4. Colab: run `batch_process.ipynb` segment audio files from each part → `PROJECT_DIR/segments/partN`
# 5. Colab: run `merge_segment_parts.ipynb` to merge all segment folders → `PROJECT_DIR/segments/all_files`
# 6. Colab: run `normalization_and_hf_push.ipynb` to text normalization → HF dataset creation and upload

In [ ]:
#CELL1 - Mount Google Drive (normalization)
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#CELL2 - Install Dependencies (normalization)
# Install required packages
!pip install num2words -q

In [ ]:
#CELL3 - Imports and Configuration (normalization)
import os
import re
from pathlib import Path
from num2words import num2words
from tqdm import tqdm
import shutil

# Settings
# Project root — folder under MyDrive (must match batch notebook)
PROJECT_DIR = "/content/drive/MyDrive/project_name/segments"  # edit path

# Segment folder to normalize (batch notebook output)
AUDIO_DIR = f"{PROJECT_DIR}/all_files"  # edit path — part1, part2, ... or all_files if you used merge_segment_parts.ipynb

# Backup folder (created automatically on first run)
BACKUP_DIR = f"{PROJECT_DIR}/backup/segments/all_files"  # edit path — must match AUDIO_DIR

NORMALIZE_LANG = "en"  # num2words language code

In [ ]:
#CELL4 - Backup Function (normalization)
# Backup helper
def create_backup():
    """Back up original files before normalization."""
    if not os.path.exists(BACKUP_DIR):
        print(f"Creating backup: {BACKUP_DIR}")
        shutil.copytree(AUDIO_DIR, BACKUP_DIR)
        print("✓ Backup complete!")
    else:
        print("⚠ Backup folder already exists, skipping backup.")

In [ ]:
#CELL5 - Text Normalization Functions (used by CELL7 only; HF push reads result as-is)


# Subtitle pattern — flexible match
# "altyazı" or "altyazılı" + space/punctuation + letter + dot + letter + dot
# Whisper occasionally hallucinates and appends a subtitle credit (e.g., "altyazı by XX") to the end of the transcript, even when it is not present in the audio.
# Clean such artifacts using patterns appropriate for the transcript language.
SUBTITLE_PATTERN = re.compile(
    r'altyaz[ıi]l?[ıi]?\s*[a-zğüşıöçA-ZĞÜŞİÖÇ]\s*\.\s*[a-zğüşıöçA-ZĞÜŞİÖÇ]\s*\.?',
    re.IGNORECASE
)

def contains_only_subtitle(text):
    """Return True if text contains only subtitle credit lines."""
    cleaned = text.strip()

    # Find and remove subtitle pattern
    temp_text = SUBTITLE_PATTERN.sub('', cleaned).strip()

    # Strip remaining punctuation and whitespace
    temp_text = re.sub(r'[.,;:!?\s\n\r]+', '', temp_text)

    # Nothing left — subtitle only
    return len(temp_text) == 0

def remove_subtitle_text(text):
    """Remove subtitle credit lines from text."""
    result = SUBTITLE_PATTERN.sub('', text).strip()
    # Collapse extra whitespace and punctuation
    result = re.sub(r'\s+', ' ', result)
    result = re.sub(r'\s*\.\s*$', '.', result)  # Trim trailing spaces before final dot
    return result

def normalize_quotes(text):
    """Normalize quote characters to standard double quotes."""
    # Convert '', "", and other quote variants to "
    text = text.replace("''", '"')
    text = text.replace("''", '"')
    text = text.replace(""", '"')
    text = text.replace(""", '"')
    text = text.replace("„", '"')
    text = text.replace("‟", '"')
    return text

def convert_numbers_to_words(text):
    """Convert numbers to words using NORMALIZE_LANG."""

    # Ordinals (1., 2., 3.) — must be at word boundary
    def ordinal_replacement(match):
        num = int(match.group(1))
        try:
            return num2words(num, lang=NORMALIZE_LANG, to='ordinal')
        except:
            return match.group(0)

    # Convert ordinals (e.g. "1.", "2.") — only when followed by space or EOL
    text = re.sub(r'\b(\d+)\.(?=\s|$)', ordinal_replacement, text)

    # Decimals (25,5 or 25.5)
    if NORMALIZE_LANG == "tr":
        def decimal_replacement(match):
            num_str = match.group(0)
            try:
                # Turkish decimal separator is comma; accept both comma and dot
                num_str_normalized = num_str.replace('.', ',')
                parts = num_str_normalized.split(',')

                if len(parts) == 2:
                    integer_part = num2words(int(parts[0]), lang=NORMALIZE_LANG)
                    # Read decimal digits one by one
                    decimal_digits = [num2words(int(d), lang=NORMALIZE_LANG) for d in parts[1]]
                    decimal_part = ' '.join(decimal_digits)
                    return f"{integer_part} virgül {decimal_part}"
                else:
                    return num2words(int(parts[0]), lang=NORMALIZE_LANG)
            except:
                return match.group(0)
    else:
        def decimal_replacement(match):
            num_str = match.group(0).replace(',', '.')
            try:
                return num2words(float(num_str), lang=NORMALIZE_LANG)
            except:
                return match.group(0)

    text = re.sub(r'\b\d+[.,]\d+\b', decimal_replacement, text)

    # Convert plain integers
    def number_replacement(match):
        try:
            num = int(match.group(0))
            return num2words(num, lang=NORMALIZE_LANG)
        except:
            return match.group(0)

    text = re.sub(r'\b\d+\b', number_replacement, text)

    return text

def normalize_text(text):
    """Normalize transcript text."""
    original_text = text

    if NORMALIZE_LANG == "tr":
        # Subtitle-only text → None (file pair will be deleted)
        if contains_only_subtitle(text):
            return None

        # Strip subtitle lines when other content remains
        text = remove_subtitle_text(text)

    # Empty after cleanup → None
    if not text or text.strip() == '':
        return None

    # Normalize quotes
    text = normalize_quotes(text)

    # Convert numbers to words
    text = convert_numbers_to_words(text)

    # Collapse extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [ ]:
#CELL6 - Process Files Function (normalization)
def process_files():
    """Process all txt/wav pairs in AUDIO_DIR."""

    # Find txt files
    txt_files = list(Path(AUDIO_DIR).glob("*.txt"))

    if not txt_files:
        print(f"❌ No .txt files found in {AUDIO_DIR}!")
        return

    print(f"📊 Found {len(txt_files)} txt files total.")

    stats = {
        'processed': 0,
        'deleted': 0,
        'cleaned': 0,
        'unchanged': 0,
        'errors': 0
    }

    deleted_files = []
    cleaned_files = []

    # Process each txt file
    for txt_path in tqdm(txt_files, desc="Processing"):
        try:
            # Read txt file
            with open(txt_path, 'r', encoding='utf-8') as f:
                original_text = f.read()

            # Normalize text
            normalized_text = normalize_text(original_text)

            # Delete pair if subtitle-only or empty
            if normalized_text is None:
                # Matching wav file
                wav_path = txt_path.with_suffix('.wav')

                # Delete both files
                deleted = False
                if txt_path.exists():
                    txt_path.unlink()
                    deleted = True
                if wav_path.exists():
                    wav_path.unlink()

                if deleted:
                    stats['deleted'] += 1
                    deleted_files.append(txt_path.name)

            # Update txt if text changed
            elif normalized_text != original_text:
                with open(txt_path, 'w', encoding='utf-8') as f:
                    f.write(normalized_text)
                stats['cleaned'] += 1
                cleaned_files.append((txt_path.name, original_text[:50], normalized_text[:50]))
            else:
                stats['unchanged'] += 1

            stats['processed'] += 1

        except Exception as e:
            print(f"\n❌ Hata ({txt_path.name}): {str(e)}")
            stats['errors'] += 1

    # Print summary stats
    print("\n" + "="*60)
    print("📈 PROCESSING RESULTS")
    print("="*60)
    print(f"✓ Files processed: {stats['processed']}")
    print(f"✓ Texts cleaned: {stats['cleaned']}")
    print(f"✓ File pairs deleted: {stats['deleted']}")
    print(f"→ Unchanged files: {stats['unchanged']}")
    print(f"✗ Errors: {stats['errors']}")
    print("="*60)

    # Sample deleted files
    if deleted_files:
        print(f"\n🗑️  Deleted files (first 10):")
        for fname in deleted_files[:10]:
            print(f"   - {fname}")
        if len(deleted_files) > 10:
            print(f"   ... and {len(deleted_files) - 10} more files")

    # Sample cleaned files
    if cleaned_files:
        print(f"\n🧹 Cleaned files (first 5):")
        for fname, orig, new in cleaned_files[:5]:
            print(f"\n   📄 {fname}")
            print(f"      Before: {orig}...")
            print(f"      After: {new}...")


In [ ]:
#CELL7 - Run Normalization
# Main entry point
print("="*60)
print("🎙️  TTS TEXT NORMALIZATION AND CLEANUP")
print("="*60)

# Create backup
print("\n2️⃣  Creating backup...")
create_backup()

# Run normalization
print("\n3️⃣  Processing files...")
print(f"📁 Directory: {AUDIO_DIR}")
process_files()

print("\n✅ Processing complete!")
print(f"💾 Backup folder: {BACKUP_DIR}")


In [ ]:
# --- hf-push ---

#CELL8 - Install & Imports (hf-push)
!pip install torch torchaudio datasets pandas huggingface_hub tqdm torchcodec num2words

import os
import pandas as pd
import torchaudio
import torchcodec
import torchaudio.transforms as T
from datasets import Dataset, Audio
from huggingface_hub import login
import torch
from tqdm.auto import tqdm
import gc
import pickle
import json
from pathlib import Path
import re
from num2words import num2words

print("✓ Imports successful!")



In [ ]:
#CELL9 - Configuration (hf-push)
# Fill in the values below, then run this cell.

# ===== CONFIGURATION =====
# HuggingFace: https://huggingface.co/settings/tokens
HF_USERNAME = ""  # enter your HuggingFace username
HF_TOKEN = ""  # enter your — hf_... token

# Dataset name on the Hub → result: {HF_USERNAME}/{OUTPUT_DATASET_NAME}
OUTPUT_DATASET_NAME = ""  # enter dataset name — e.g. tts-dataset

# Project root
PROJECT_DIR = "/content/drive/MyDrive/project_name"  # edit path

# Audio source mode:
#   "all_files" — use merged folder after merge_segment_parts.ipynb (recommended)
#   "parts"     — scan part1, part2, part3 under AUDIO_DIR (skip merge)
AUDIO_SOURCE_MODE = "all_files"  # edit path

# Base path for audio discovery
# all_files mode → reads PROJECT_DIR/segments/all_files (flat 1.wav, 1.txt, ...)
# parts mode     → reads PROJECT_DIR/segments/part1, part2, ...
AUDIO_DIR = f"{PROJECT_DIR}/segments"  # edit path
# =========================

# Checkpoint directory (Colab local storage - fast)
CHECKPOINT_DIR = "/content/checkpoints"
PROCESSED_AUDIO_DIR = "/content/processed_audio"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PROCESSED_AUDIO_DIR, exist_ok=True)

# Save config for other cells
config = {
    "HF_USERNAME": HF_USERNAME,
    "HF_TOKEN": HF_TOKEN,
    "OUTPUT_DATASET_NAME": OUTPUT_DATASET_NAME,
    "AUDIO_DIR": AUDIO_DIR,
    "AUDIO_SOURCE_MODE": AUDIO_SOURCE_MODE,
    "CHECKPOINT_DIR": CHECKPOINT_DIR,
    "PROCESSED_AUDIO_DIR": PROCESSED_AUDIO_DIR
}

with open(f"{CHECKPOINT_DIR}/config.json", "w") as f:
    json.dump(config, f)

print("✓ Configuration saved!")
print(f"  Dataset name: {OUTPUT_DATASET_NAME}")
print(f"  Audio source: {AUDIO_SOURCE_MODE}")
print(f"  Audio dir: {AUDIO_DIR}")
print(f"  Checkpoint dir: {CHECKPOINT_DIR}")

In [ ]:
#CELL10 - Mount Drive & Discover Audio Files (hf-push)
from google.colab import drive

# Load config
with open(f"{CHECKPOINT_DIR}/config.json", "r") as f:
    config = json.load(f)

AUDIO_DIR = config["AUDIO_DIR"]
AUDIO_SOURCE_MODE = config.get("AUDIO_SOURCE_MODE", "all_files")
CHECKPOINT_DIR = config["CHECKPOINT_DIR"]

# Mount drive
print("Mounting Google Drive...")
drive.mount('/content/drive')
print("✓ Drive mounted!")

def discover_audio_files_from_merged(base_dir, merged_folder="all_files"):
    """
    Find wav/txt pairs in the merged flat folder (1.wav, 2.wav, ...).
    Use after merge_segment_parts.ipynb.
    """
    merged_path = os.path.join(base_dir, merged_folder)
    if not os.path.exists(merged_path):
        print(f"Error: Merged folder not found: {merged_path}")
        return []

    wav_files = [f for f in os.listdir(merged_path) if f.endswith('.wav')]
    try:
        wav_files.sort(key=lambda x: int(os.path.splitext(x)[0]))
    except ValueError:
        wav_files.sort()

    all_files = []
    for wav_file in wav_files:
        base_name = os.path.splitext(wav_file)[0]
        original_path = os.path.join(merged_path, wav_file)
        all_files.append({
            'original_path': original_path,
            'unique_name': wav_file,
            'part_folder': merged_folder,
            'original_name': wav_file
        })

    print(f"  Found {len(all_files)} files in {merged_path}")
    return all_files

# Discover audio files from multiple part folders
def discover_audio_files_from_parts(base_dir, part_folders=None):
    """
    Find all wav files under part1, part2, part3 folders.
    Assign unique names: part1_001.wav, part2_001.wav
    Use only if you skipped the merge step.
    """
    if not os.path.exists(base_dir):
        print(f"Error: Directory not found: {base_dir}")
        return []

    if part_folders is None:
        part_folders = ['part1', 'part2', 'part3']

    all_files = []

    for part in part_folders:
        part_path = os.path.join(base_dir, part)
        if not os.path.exists(part_path):
            print(f"⚠️  Warning: {part} folder not found, skipping...")
            continue

        wav_files = sorted([f for f in os.listdir(part_path) if f.endswith('.wav')])
        print(f"  Found {len(wav_files)} files in {part}")

        for wav_file in wav_files:
            original_path = os.path.join(part_path, wav_file)
            base_name = os.path.splitext(wav_file)[0]
            # Unique name: part1_001.wav, etc.
            unique_name = f"{part}_{base_name}.wav"
            all_files.append({
                'original_path': original_path,
                'unique_name': unique_name,
                'part_folder': part,
                'original_name': wav_file
            })

    return all_files

if AUDIO_SOURCE_MODE == "all_files":
    print("Discovering audio from merged all_files folder...")
    audio_files = discover_audio_files_from_merged(AUDIO_DIR)
    error_hint = f"{AUDIO_DIR}/all_files"
else:
    print("Discovering audio from part1, part2, part3 folders...")
    audio_files = discover_audio_files_from_parts(AUDIO_DIR)
    error_hint = f"{AUDIO_DIR}/part1,part2,part3"

if not audio_files:
    print(f"ERROR: No .wav files found in {error_hint}")
else:
    print(f"\n✓ Found {len(audio_files)} audio files total")

    # Save audio file list
    with open(f"{CHECKPOINT_DIR}/audio_files.pkl", "wb") as f:
        pickle.dump(audio_files, f)

    print(f"✓ Audio file list saved to checkpoint!")



In [ ]:
#CELL11 - Audio Processing Functions
def soft_limiter(waveform, threshold=0.95, knee=0.05):
    abs_wave = torch.abs(waveform)
    gain = torch.ones_like(waveform)
    mask = abs_wave > threshold
    if mask.any():
        excess = abs_wave[mask] - threshold
        compressed = threshold + knee * torch.tanh(excess / knee)
        gain[mask] = compressed / abs_wave[mask]
    return waveform * gain


def normalize_loudness(waveform, target_lufs=-23.0):
    rms = torch.sqrt(torch.mean(waveform ** 2))
    if rms < 1e-8:
        return waveform
    target_rms = 10 ** (target_lufs / 20)
    normalized = waveform * (target_rms / rms)
    normalized = soft_limiter(normalized, threshold=0.95, knee=0.05)
    max_val = torch.max(torch.abs(normalized))
    if max_val > 0.99:
        normalized = normalized * (0.99 / max_val)
    return normalized


def resample_and_normalize_audio(audio_path, target_sr=24000, output_dir=None, unique_filename=None):
    try:
        waveform, sample_rate = torchaudio.load(audio_path)

        # Move to GPU if available
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        waveform = waveform.to(device)

        if sample_rate != target_sr:
            resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sr).to(device)
            waveform = resampler(waveform)

        waveform = normalize_loudness(waveform)

        # Move back to CPU for saving
        waveform = waveform.cpu()

        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            filename = unique_filename if unique_filename else os.path.basename(audio_path)
            output_path = os.path.join(output_dir, filename)
            torchaudio.save(output_path, waveform, target_sr)
            return output_path

        return audio_path
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None

print("✓ Audio processing functions defined!")


In [ ]:
#CELL12 - Process Audio Files (WITH CHECKPOINTS)
# Transcripts are already normalized on Drive by CELL7 — read txt as-is.
# Load config and audio files
with open(f"{CHECKPOINT_DIR}/config.json", "r") as f:
    config = json.load(f)

with open(f"{CHECKPOINT_DIR}/audio_files.pkl", "rb") as f:
    audio_files = pickle.load(f)

AUDIO_DIR = config["AUDIO_DIR"]
CHECKPOINT_DIR = config["CHECKPOINT_DIR"]
PROCESSED_AUDIO_DIR = config["PROCESSED_AUDIO_DIR"]

# Load existing progress if any
progress_file = f"{CHECKPOINT_DIR}/processing_progress.pkl"
if os.path.exists(progress_file):
    with open(progress_file, "rb") as f:
        checkpoint_data = pickle.load(f)
        all_data = checkpoint_data["all_data"]
        processed_files = checkpoint_data["processed_files"]
        stats = checkpoint_data.get("stats", {
            'total_processed': 0,
            'valid_entries': 0,
            'skipped_no_text': 0,
            'skipped_empty': 0,
            'skipped_audio_error': 0
        })
    print(f"✓ Resuming from checkpoint: {len(processed_files)} files already processed")
else:
    all_data = []
    processed_files = set()
    stats = {
        'total_processed': 0,
        'valid_entries': 0,
        'skipped_no_text': 0,
        'skipped_empty': 0,
        'skipped_audio_error': 0
    }
    print("Starting fresh processing...")

# Process each audio file
total_files = len(audio_files)
print(f"\nProcessing {total_files} audio files...")

for idx, audio_info in enumerate(tqdm(audio_files, desc="Processing audio")):
    # Skip if already processed
    unique_id = f"{audio_info['part_folder']}_{audio_info['original_name']}"
    if unique_id in processed_files:
        continue

    # Original file paths
    audio_path = audio_info['original_path']
    base_name = os.path.splitext(audio_info['original_name'])[0]
    part_path = os.path.dirname(audio_path)
    transcript_path = os.path.join(part_path, f"{base_name}.txt")

    stats['total_processed'] += 1

    # Check transcript file exists
    if not os.path.exists(transcript_path):
        stats['skipped_no_text'] += 1
        processed_files.add(unique_id)
        continue

    try:
        # Read normalized transcript (CELL7)
        with open(transcript_path, 'r', encoding='utf-8') as f:
            text = f.read().strip()

        if not text:
            stats['skipped_empty'] += 1
            processed_files.add(unique_id)
            continue

        # Process audio — save with unique filename
        processed_audio_path = resample_and_normalize_audio(
            audio_path,
            target_sr=24000,
            output_dir=PROCESSED_AUDIO_DIR,
            unique_filename=audio_info['unique_name']
        )

        if processed_audio_path:
            all_data.append({
                "audio_path": processed_audio_path,
                "text": text
            })
            stats['valid_entries'] += 1
        else:
            stats['skipped_audio_error'] += 1

        processed_files.add(unique_id)

    except Exception as e:
        print(f"\n  Error processing {audio_info['unique_name']}: {e}")
        stats['skipped_audio_error'] += 1
        processed_files.add(unique_id)

    # Save checkpoint every 100 files
    if (idx + 1) % 100 == 0:
        checkpoint_data = {
            "all_data": all_data,
            "processed_files": processed_files,
            "stats": stats
        }
        with open(progress_file, "wb") as f:
            pickle.dump(checkpoint_data, f)
        print(f"\n  💾 Checkpoint saved!")
        print(f"     Processed: {stats['total_processed']}/{total_files}")
        print(f"     Valid entries: {stats['valid_entries']}")
        print(f"     Skipped: {stats['skipped_no_text'] + stats['skipped_empty'] + stats['skipped_audio_error']}")
        gc.collect()

# Final checkpoint save
checkpoint_data = {
    "all_data": all_data,
    "processed_files": processed_files,
    "stats": stats
}
with open(progress_file, "wb") as f:
    pickle.dump(checkpoint_data, f)

print(f"\n{'='*60}")
print(f"✓ ALL FILES PROCESSED!")
print(f"{'='*60}")
print(f"  Total files processed: {stats['total_processed']}")
print(f"  ✓ Valid entries: {stats['valid_entries']}")
print(f"  ⊗ Skipped (no text file): {stats['skipped_no_text']}")
print(f"  ⊗ Skipped (empty text): {stats['skipped_empty']}")
print(f"  ⊗ Skipped (audio error): {stats['skipped_audio_error']}")
print(f"  📊 Success rate: {stats['valid_entries']}/{stats['total_processed']} ({100*stats['valid_entries']/max(stats['total_processed'],1):.1f}%)")
print(f"{'='*60}")
print(f"\n✅  Original files NOT modified - they remain safe in Google Drive!")



In [ ]:
#CELL13 - Create Dataset
# Load config and processed data
with open(f"{CHECKPOINT_DIR}/config.json", "r") as f:
    config = json.load(f)

with open(f"{CHECKPOINT_DIR}/processing_progress.pkl", "rb") as f:
    checkpoint_data = pickle.load(f)
    all_data = checkpoint_data["all_data"]

print(f"Creating dataset from {len(all_data)} entries...")

# Create DataFrame
df = pd.DataFrame(all_data)

# Create HF Dataset
print("Converting to HuggingFace Dataset format...")
dataset = Dataset.from_pandas(df)

print("Casting audio column...")
dataset = dataset.cast_column("audio_path", Audio())
dataset = dataset.rename_column("audio_path", "audio")

# Save dataset locally
OUTPUT_DATASET_NAME = config["OUTPUT_DATASET_NAME"]
local_path = f"/content/{OUTPUT_DATASET_NAME}-dataset"

print(f"Saving dataset to {local_path}...")
dataset.save_to_disk(local_path)

# Save dataset path to checkpoint
with open(f"{CHECKPOINT_DIR}/dataset_path.txt", "w") as f:
    f.write(local_path)

print(f"\n✓ Dataset created and saved!")
print(f"  Total samples: {len(dataset)}")
print(f"  Path: {local_path}")



In [ ]:
#CELL14 - Upload to HuggingFace
# Load config
with open(f"{CHECKPOINT_DIR}/config.json", "r") as f:
    config = json.load(f)

with open(f"{CHECKPOINT_DIR}/dataset_path.txt", "r") as f:
    local_path = f.read().strip()

HF_USERNAME = config["HF_USERNAME"]
HF_TOKEN = config["HF_TOKEN"]
OUTPUT_DATASET_NAME = config["OUTPUT_DATASET_NAME"]

# Login to HuggingFace
print("Logging in to HuggingFace Hub...")
login(token=HF_TOKEN)

# Load dataset
print(f"Loading dataset from {local_path}...")
dataset = Dataset.load_from_disk(local_path)

# Push to hub
full_dataset_name = f"{HF_USERNAME}/{OUTPUT_DATASET_NAME}"
print(f"\nUploading to HuggingFace Hub as {full_dataset_name}...")
print("This may take a while for large datasets...")

dataset.push_to_hub(
    full_dataset_name,
    private=False,
)

print(f"\n{'='*60}")
print(f"✓✓✓ SUCCESS! Dataset uploaded to HuggingFace!")
print(f"{'='*60}")
print(f"  Dataset: {full_dataset_name}")
print(f"  Samples: {len(dataset)}")
print(f"  TTS_dataset: '{full_dataset_name}'")
print(f"{'='*60}")
